<a href="https://colab.research.google.com/github/ZanebRA/urdu-ocr-codesaviours-si26-zaneb/blob/main/SI26_Week5_Zaneb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers gradio torch pillow sentencepiece

In [2]:
import gradio
import transformers
import torch

print("All packages installed successfully!")

All packages installed successfully!


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model_path = "/content/drive/MyDrive/trocr_urdu_model"

processor = TrOCRProcessor.from_pretrained(model_path)
model = VisionEncoderDecoderModel.from_pretrained(model_path)

print("✅ Model loaded successfully!")

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

✅ Model loaded successfully!


In [5]:
import gradio as gr
from PIL import Image
import torch

model.eval()

def extract_urdu_text(image):

    if image is None:
        return "Please upload an image."

    pixel_values = processor(image, return_tensors="pt").pixel_values

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    if text.strip() == "":
        return "Could not extract text."

    return text


interface = gr.Interface(
    fn=extract_urdu_text,
    inputs=gr.Image(type="pil", label="Upload Urdu Image"),
    outputs=gr.Textbox(label="Extracted Urdu Text"),
    title="Urdu OCR - Code Saviours SI-26",
    description="Upload an Urdu image to extract text."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a50f7ce3b22493e51.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
!ls /content/drive/MyDrive

 Classroom	    labels.gsheet	 week4_ML_SI-26.gdoc
'Colab Notebooks'  'Saved from Chrome'	 week4_ML_SI-26.pdf
 data		    trocr_urdu_model	 zaneb-rasool-ahmed.pdf
 labels.csv.txt     UNHD-Complete-Data


In [10]:
!ls /content/drive/MyDrive/trocr_urdu_model

config.json		model.safetensors	  tokenizer_config.json
generation_config.json	preprocessor_config.json  tokenizer.json
merges.txt		special_tokens_map.json   vocab.json


In [13]:
!du -sh /content/drive/MyDrive/trocr_urdu_model

1.3G	/content/drive/MyDrive/trocr_urdu_model


In [14]:
!find /content/drive/MyDrive/trocr_urdu_model -name "*.safetensors" -o -name "*.bin"

/content/drive/MyDrive/trocr_urdu_model/model.safetensors


In [15]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 43.2 MB/s eta 0:00:00


In [16]:
%%writefile app.py

import streamlit as st
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import torch

MODEL_PATH = "/content/drive/MyDrive/trocr_urdu_model"

processor = TrOCRProcessor.from_pretrained(MODEL_PATH)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

st.title("Urdu OCR - Code Saviours SI-26")
st.write("Upload an Urdu image to extract text.")

uploaded_file = st.file_uploader("Choose an Urdu image", type=["png", "jpg", "jpeg"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded Image", use_container_width=True)

    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    st.subheader("Extracted Text")
    st.write(text)

Writing app.py


In [17]:
%%writefile requirements.txt

streamlit
transformers
torch
pillow
sentencepiece
tiktoken
safetensors

Writing requirements.txt


In [18]:
!streamlit run app.py &>/content/logs.txt &

In [20]:
!ls

app.py	drive  logs.txt  requirements.txt  sample_data


In [21]:
%%writefile README.md

# Urdu OCR - Code Saviours SI-26

This project is an Urdu Optical Character Recognition (OCR) system developed using Microsoft's TrOCR model.

## Features
- Upload an Urdu image
- Extract Urdu text automatically
- Built with Streamlit and Transformers

## Requirements
- Python 3.10+
- Streamlit
- Transformers
- Torch
- Pillow

## Run

```bash
streamlit run app.py

Writing README.md
